##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Analyze a Video - Historic Event Recognition

This notebook shows how you can use Gemini models' multimodal capabilities to recognize which historic event is happening in the video.

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Analyze_a_Video_Historic_Event_Recognition.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/137.7 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━ 133.1/137.7 kB 5.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 2.6 MB/s eta 0:00:00


In [ ]:
%pip install -U -q "google-genai>=1.0.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 416.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.8/728.8 kB 13.3 MB/s eta 0:00:00


## Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.

In [ ]:
#from google import genai
#from google.colab import userdata

#API_KEY = userdata.get('GOOGLE_API_KEY')
#client = genai.Client(api_key=API_KEY)

In [ ]:
from google import genai
from google.colab import userdata
API_KEY=userdata.get('GOOGLE_API_KEY2')
client=genai.Client(api_key=API_KEY)


## Example

This example uses [video of President Ronald Reagan's Speech at the Berlin Wall](https://s3.amazonaws.com/NARAprodstorage/opastorage/live/16/147/6014716/content/presidential-libraries/reagan/5730544/6-12-1987-439.mp4) taken on June 12 1987.

In [ ]:
# Download video
#path = "berlin.mp4"
#url = "https://s3.amazonaws.com/NARAprodstorage/opastorage/live/16/147/6014716/content/presidential-libraries/reagan/5730544/6-12-1987-439.mp4"
#!wget $url -O $path

--2025-03-04 14:04:54--  https://s3.amazonaws.com/NARAprodstorage/opastorage/live/16/147/6014716/content/presidential-libraries/reagan/5730544/6-12-1987-439.mp4
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.89.118, 52.217.64.198, 52.216.106.21, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.89.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 628645171 (600M) [video/mp4]
Saving to: ‘berlin.mp4’

berlin.mp4          100%[===================>] 599.52M  41.0MB/s    in 12s     

2025-03-04 14:05:07 (48.2 MB/s) - ‘berlin.mp4’ saved [628645171/628645171]



In [ ]:
import os

!pip install yt-dlp


# Download video using yt-dlp. It will save the file to /content/ with its original name.
!yt-dlp https://www.youtube.com/watch?v=xBU0hE_o04Y

# Define the original downloaded file name (as yt-dlp saves it)
original_downloaded_name = "Discurs Corneliu Zelea Codreanu 1933 (Declarația din parlament). [xBU0hE_o04Y].mp4"
original_path = os.path.join("/content", original_downloaded_name)

# Define a simpler, ASCII-only path for the video
new_path_name = "codreanu_speech.mp4"
new_path = os.path.join("/content", new_path_name)

# Rename the file to avoid issues with special characters during upload
# Check if the original file exists before renaming
if os.path.exists(original_path):
  !mv "$original_path" "$new_path"
  print(f"Renamed '{original_downloaded_name}' to '{new_path_name}'")
else:
  print(f"Original file '{original_downloaded_name}' not found. Assuming it was already renamed or not downloaded.")

# Set the 'path' variable to the new, simplified filename for subsequent cells
path = new_path
print(f"'path' variable set to: {path}")

[youtube] Extracting URL: https://www.youtube.com/watch?v=xBU0hE_o04Y
[youtube] xBU0hE_o04Y: Downloading webpage
[youtube] xBU0hE_o04Y: Downloading android vr player API JSON
[info] xBU0hE_o04Y: Downloading 1 format(s): 18
[download] Destination: Discurs Corneliu Zelea Codreanu 1933 (Declarația din parlament). [xBU0hE_o04Y].mp4
[download] 100% of   17.40MiB in 00:00:01 at 15.30MiB/s
Renamed 'Discurs Corneliu Zelea Codreanu 1933 (Declarația din parlament). [xBU0hE_o04Y].mp4' to 'codreanu_speech.mp4'
'path' variable set to: /content/codreanu_speech.mp4


In [ ]:
#Upload video
js_runtimes:{'deno':{'path':None},'node':{'path':'C:/Program Files/nodejs/node.exe'}}
video_file=client.files.upload(file=path)

In [ ]:
import time
# Wait until the uploaded video is available
while video_file.state.name == "PROCESSING":
  print('.', end='')
  time.sleep(5)
  video_file = client.files.get(name=video_file.name)

if video_file.state.name == "FAILED":
  raise ValueError(video_file.state.name)

.

The uploaded video is ready for processing. This prompt instructs the model to provide basic information about the historical events portrayed in the video.

In [ ]:
system_prompt = """
  You are historian who specializes in events caught on film.
  When you receive a video answer following questions:
  When did it happen?
  Who is the most important person in video?
  How the event is called?
"""

Some historic events touch on controversial topics that may get flagged by Gemini API, which blocks the response for the query.

Because of this, it might be a good idea to turn off safety settings.

In [ ]:
safety_settings = [
    {
        "category": "HARM_CATEGORY_HARASSMENT",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_HATE_SPEECH",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
        "threshold": "BLOCK_NONE",
    },
]

In [ ]:
from google.genai import types

MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}
response = client.models.generate_content(
    model=f"models/{MODEL_ID}",
    contents=[
        "Analyze the video please",
        video_file
        ],
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        safety_settings=safety_settings,
        ),
    )
print(response.text)

Sure, here's the information about the video:

- **When did it happen?** It was in 1933.
- **Who is the most important person in the video?** Corneliu Zelea Codreanu. He was a Romanian politician and the founder of the Legion of the Archangel Michael.
- **How the event is called?** This event is called the Declaration of 1933. It was a programmatic manifesto where Codreanu outlined the goals and principles of his political movement.


In [ ]:
# Delete video
client.files.delete(name=video_file.name)

DeleteFileResponse()

## Summary

Now you know how you can prompt Gemini models with videos and use them to recognize historic events.

This notebook shows only one of many use cases. Check the [Video understanding](../quickstarts/Video_understanding.ipynb) notebook for more examples of using the Gemini API with videos.